# NBL Cell Clustering

## Libraries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import anndata_repr  # noqa: F401
import buckaroo  # noqa: F401
import pandas as pd
import natsort as ns
import bionty as bt
import lamindb as ln
import nbl
from distributed import as_completed  # noqa: F401
import spatialdata as sd
from nbl._util import DaskLocalCluster
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import squidpy as sq

In [ ]:
pd.set_option("future.no_silent_downcasting", True)
pd.set_option("mode.copy_on_write", True)

In [ ]:
cluster = DaskLocalCluster(n_workers=10, threads_per_worker=1)
cluster(open_dashboard=True)

In [ ]:
# ln.settings.transform.stem_uid = "XRQzD0XFnmQv"
# ln.settings.transform.version = "1"
# ln.track()

## Create FeatureSet from Immune Markers

In [ ]:
cell_marker_lookup = bt.CellMarker.lookup()

In [ ]:
immune_feature_set = ln.FeatureSet(
    features=[
        cell_marker_lookup.calprotectin,
        cell_marker_lookup.cd11b,
        cell_marker_lookup.cd11c,
        cell_marker_lookup.cd14,
        cell_marker_lookup.cd15,
        cell_marker_lookup.cd16,
        cell_marker_lookup.cd166,
        cell_marker_lookup.cd206,
        cell_marker_lookup.cd209,
        cell_marker_lookup.cd3,
        cell_marker_lookup.cd4,
        cell_marker_lookup.cd45,
        cell_marker_lookup.cd56,
        cell_marker_lookup.cd57,
        cell_marker_lookup.cd68,
        cell_marker_lookup.cd8,
    ],
    name="Immune Markers",
)
immune_feature_set.save()

In [ ]:
immune_markers = ns.natsorted([im.name for im in ln.FeatureSet.lookup().immune_markers.members])

## Load Data

In [ ]:
nbl_data_path = Path("../../../data/nbl_data/nbl.ome.zarr")

In [ ]:
nbl_sdata = sd.read_zarr(store=nbl_data_path)

### Figure Paths

In [ ]:
fig_dir = Path("/Users/srivarra/Davis Lab/NBL/data/plots/nbl_markers/figures")

### Filter NBL Cells and Immune Markers

In [ ]:
nbl_sdata.tables["whole_cell"].strings_to_categoricals()

In [ ]:
wc_adata: ad.AnnData = nbl_sdata.tables["whole_cell"]

In [ ]:
wc_adata.layers["area_normalized"] = (wc_adata.X / wc_adata.obs["area"].to_numpy()[:, None]).todense()

In [ ]:
wc_adata.shape

In [ ]:
wc_adata.to_df(layer="area_normalized")

In [ ]:
nbl_cell_idx = wc_adata.obs.pixie_cluster.isin(["NBL_cell"])

In [ ]:
nbl_cell_immune_adata = wc_adata[nbl_cell_idx, immune_markers]

In [ ]:
nbl_sdata.tables["nbl_cell-immune_markers"] = nbl_cell_immune_adata

In [ ]:
nbl_immune_adata = nbl_sdata.tables["nbl_cell-immune_markers"]

### FlowSOM Clusters

In [ ]:
adata_flowsom, cluster_objs = nbl.tl.flowsom_clustering(
    adata=nbl_immune_adata,
    layer="arcsinh_cf_0.05",
    key_added="flowsom_clusters",
    som_dim=(10, 10),
    min_clusters=2,
    max_clusters=10,
    agglomerative_clustering_kwargs={"metric": "euclidean", "linkage": "ward"},
    return_clustering_objs=True,
)

In [ ]:
fig = plt.figure(figsize=(6, 16), dpi=300, layout="constrained")

fig.suptitle(t=r"FlowSOM Clusters -- Choosing the Best $K$")
auc_cdf_ax, cdf_ax = fig.subplots(nrows=2, ncols=1)

cluster_objs["consensus_clustering"].plot_auc_cdf(ax=auc_cdf_ax)
cluster_objs["consensus_clustering"].plot_cdf(ax=cdf_ax)
fig.savefig(fig_dir / "flowsom_choosing_K.pdf")

In [ ]:
clustermap_fig = cluster_objs["consensus_clustering"].plot_clustermap(k=10, figsize=(16, 16))

clustermap_fig.savefig(fname=fig_dir / "FlowSOM_clustermap.pdf")

In [ ]:
sc.pp.pca(nbl_immune_adata, layer="arcsinh_cf_0.05")

In [ ]:
# PCA Figure

pca_fig = plt.figure(figsize=(16, 16), layout="constrained", dpi=300)

pca_subfigs = pca_fig.subplots(nrows=1, ncols=1)
sc.pl.pca(
    adata=nbl_cell_immune_adata,
    layer="arcsinh_cf_0.05",
    color="flowsom_clusters",
    annotate_var_explained=True,
    ax=pca_subfigs,
    size=10,
)
pca_fig.savefig(fname=fig_dir / "pca_scatter.pdf")

### Leiden Clustering

In [ ]:
nbl_cell_immune_adata = nbl_sdata.tables["nbl_cell-immune_markers"]

In [ ]:
sq.gr.spatial_neighbors(
    adata=nbl_cell_immune_adata, spatial_key="spatial", library_key="region", coord_type="generic", radius=200
)

In [ ]:
nbl_cell_immune_adata

Use the `X` PCA representation generated from the `arcsinh` transformed layer.

In [ ]:
sc.pp.neighbors(adata=nbl_cell_immune_adata, use_rep="X_pca")

In [ ]:
sc.tl.leiden(adata=nbl_cell_immune_adata, neighbors_key="neighbors")

In [ ]:
flowsom_leiden_cluster_data = sc.metrics.confusion_matrix(
    new="leiden", orig="flowsom_clusters", data=nbl_cell_immune_adata.obs
)

In [ ]:
fl_fig, fl_ax = plt.subplots(figsize=(12, 3), dpi=300, layout="constrained")
sns.heatmap(
    data=flowsom_leiden_cluster_data,
    annot=True,
    ax=fl_ax,
    square=True,
    fmt=".2f",
    annot_kws={"size": "xx-small"},
    cbar_kws={"shrink": 0.9},
    linewidths=0.01,
)
fl_ax.set_xlabel("Leiden Clusters")
fl_ax.set_ylabel("FlowSOM Clusters")
fl_fig.suptitle(t="Confusion Matrix with NBL Cell Subcluster (Immune Markers)")
fl_fig.savefig(fname=fig_dir / "nbl_cell_leiden_flowsom_immune_markers_confusion.pdf")

In [ ]:
# # nbl_sdata.tables["nbl_cell-immune_markers"]
# from nbl._util import replace_elements

# replace_elements(sdata=nbl_sdata, element_name="nbl_cell-immune_markers")

Setting cluster colors

In [ ]:
xkcd_colors_df = pd.read_csv("../../../data/cmaps/xkcd_cmaps.txt", delimiter="\t", skiprows=1)

In [ ]:
xkcd_colors_df

In [ ]:
from matplotlib.colors import to_hex

In [ ]:
pixie_colors = [
    "mauve",
    "dark purple",
    "olive",
    "sky blue",
    "teal",
    "orange",
    "salmon",
    "beige",
    "peach",
    "lilac",
    "indigo",
    "periwinkle",
    "rose",
    "rust",
]

pixie_clusters = nbl_sdata.tables["whole_cell"].obs["pixie_cluster"].cat.categories
pixie_colors_xkcd = [to_hex(f"xkcd:{c}") for c in pixie_colors]
pixie_clusters_cmap = dict(zip(pixie_clusters, pixie_colors_xkcd, strict=True))

flowsom_clusters = nbl_sdata.tables["nbl_cell-immune_markers"].obs["flowsom_clusters"].cat.categories
flowsom_colors = xkcd_colors_df.sample(n=len(flowsom_clusters))["color"]
flowsom_colors_xkcd = [to_hex(f"xkcd:{c}") for c in flowsom_colors]
flowsom_clusters_cmap = dict(zip(flowsom_clusters, flowsom_colors_xkcd, strict=False))

leiden_clusters = nbl_sdata.tables["nbl_cell-immune_markers"].obs["leiden"].cat.categories
leiden_colors = xkcd_colors_df.sample(n=len(leiden_clusters))["color"]
leiden_colors_xkcd = [to_hex(f"xkcd:{c}") for c in leiden_colors]
leiden_colors_cmap = dict(zip(leiden_clusters, leiden_colors_xkcd, strict=False))

In [ ]:
nbl_32_r3c7 = nbl_sdata.filter_by_coordinate_system(coordinate_system="NBL-32-R3C7")

In [ ]:
from dask import delayed

In [ ]:
fovs = ns.natsorted(nbl_sdata.coordinate_systems)

In [ ]:
fov_fig_dir = fig_dir / "fovs"
fov_fig_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
color_maps = {
    "pixie_cluster": pixie_clusters_cmap,
    "flowsom_clusters": flowsom_clusters_cmap,
    "leiden": leiden_colors_cmap,
}

In [ ]:
fov_sdatas = nbl.pp.split_sdata_to_fovs(sdata=nbl_sdata)

In [ ]:
fov_sdatas_futures = cluster.client.scatter(data=fov_sdatas)

In [ ]:
delayed_objs = []
for fov_sd in fov_sdatas_futures:
    f = delayed(nbl.pl.plot_subclusters)(
        sdata=fov_sd,
        fov_name=None,
        layer="arcsinh_cf_0.05",
        element_suffix="whole_cell",
        primary_table_name="whole_cell",
        secondary_table_name="nbl_cell-immune_markers",
        primary_table_obs_cols="pixie_cluster",
        seconday_table_obs_cols=["flowsom_clusters", "leiden"],
        color_maps=color_maps,
        save_dir=fov_fig_dir,
    )
    delayed_objs.append(f)

In [ ]:
cluster.client.compute(delayed_objs)

In [ ]:
nbl_32_r3c7 = nbl_sdata.filter_by_coordinate_system("NBL-32-R3C7")

In [ ]:
nbl_sdata.filter_by_coordinate_system("NBL-17-R11C10").tables["whole_cell"].obs["pixie_cluster"].value_counts()

In [ ]:
nbl.pl.plot_subclusters(
    sdata=nbl_32_r3c7,
    fov_name=None,
    layer="arcsinh_cf_0.05",
    element_suffix="whole_cell",
    primary_table_name="whole_cell",
    secondary_table_name="nbl_cell-immune_markers",
    primary_table_obs_cols="pixie_cluster",
    seconday_table_obs_cols=["flowsom_clusters", "leiden"],
    color_maps=color_maps,
);

In [ ]:
vp_fig, vp_axes = plt.subplots(nrows=1, ncols=2, figsize=(16, 6), layout="constrained", width_ratios=[6, 3], dpi=360)

leiden_sv = sc.pl.stacked_violin(
    adata=nbl_sdata.tables["nbl_cell-immune_markers"],
    var_names=immune_markers,
    layer="arcsinh_cf_0.05",
    groupby="leiden",
    ax=vp_axes[0],
    return_fig=True,
    swap_axes=True,
    title="Leiden NBL Subclusters and Immune Markers",
).add_totals(show=True, color="xkcd:cerulean")


flowsom_sv = sc.pl.stacked_violin(
    adata=nbl_sdata.tables["nbl_cell-immune_markers"],
    var_names=immune_markers,
    layer="arcsinh_cf_0.05",
    groupby="flowsom_clusters",
    ax=vp_axes[1],
    return_fig=True,
    swap_axes=True,
    title="FlowSOM Subclusters and Immune Markers",
).add_totals(show=True, color="xkcd:cerulean")

leiden_sv.color_legend_title = "Median Expression"
leiden_sv.make_figure()
flowsom_sv.color_legend_title = "Median Expression"
flowsom_sv.make_figure()

vp_fig.savefig(fig_dir / "subclusters_violin.pdf")

In [ ]:
data = sc.metrics.confusion_matrix(
    new="leiden", orig="flowsom_clusters", data=nbl_sdata.tables["nbl_cell-immune_markers"].obs, normalize=True
)
fl_fig, fl_ax = plt.subplots(figsize=(6, 2), dpi=300, layout="constrained")
sns.heatmap(
    data=data,
    annot=True,
    square=True,
    fmt="0.2f",
    annot_kws={"size": 5},
    cbar_kws={"shrink": 0.9},
    linewidths=0.01,
)
fl_ax.set_xlabel("FlowSOM Clusters")
fl_ax.set_ylabel("Leiden Clusters")
fl_fig.suptitle(t="Confusion Matrix")
fl_fig.savefig(fname=fig_dir / "nbl_cell_leiden_flowsom_immune_markers_confusion.pdf")